# 04 · Contando o estoque — **modelo genérico + especialista treinado**

Esta é a demo que amarra a palestra inteira, porque ela mostra **as duas metades
da IA aplicada** numa cena só:

| etapa | quem faz | precisou treinar? |
|---|---|---|
| achar as garrafas na foto | modelo pronto (já conhece "garrafa") | **não** |
| dizer se está aberta ou lacrada | modelo **que você ensinou** | **sim** |

> Fala de palco: *"o modelo genérico sabe o que é uma garrafa. Ele não sabe nada
> sobre o **seu** negócio. A parte que vale dinheiro é a segunda — e ela não veio
> pronta: eu ensinei separando fotos em duas pastas."*

É exatamente a mesma ideia de **agente + skill especialista**, só que em imagem.

In [ ]:
# ── 1. instala a biblioteca e monta o Google Drive ──
%pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import ultralytics, torch, os, glob
ultralytics.checks()
print("GPU disponivel:", torch.cuda.is_available())

In [ ]:
# ── ajuste de PALCO: tudo grande, porque a sala enxerga de 6 a 10 m ──
import matplotlib
matplotlib.rcParams.update({
    "figure.figsize": (16, 9),
    "figure.dpi": 110,
    "font.size": 22,
    "axes.titlesize": 30,
    "axes.labelsize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 22,
    "axes.grid": True,
    "grid.alpha": .25,
    "axes.facecolor": "#0d1117",
    "figure.facecolor": "#0d1117",
    "text.color": "#e6edf3",
    "axes.labelcolor": "#e6edf3",
    "xtick.color": "#e6edf3",
    "ytick.color": "#e6edf3",
    "axes.edgecolor": "#30363d",
    "axes.titlecolor": "#3fe0a8",
})
VERDE, VERMELHO, CINZA = "#3fe0a8", "#ff5c5c", "#7d8590"
DRIVE = "/content/drive/MyDrive/PALESTRA-IA"
print("palco configurado · raiz no Drive:", DRIVE)

## Etapa 1 — contar (modelo pronto, zero treino)

In [ ]:
import glob, cv2, matplotlib.pyplot as plt
from collections import Counter

detector = YOLO(f"{DRIVE}/00-pesos/yolo11n.pt")
CLASSE_GARRAFA = 39          # "bottle" no COCO

fotos = [f for f in sorted(glob.glob(f"{DRIVE}/04-garrafas/inferencia/*"))
         if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
if not fotos:
    raise SystemExit("coloque fotos em 04-garrafas/inferencia/ (rode o notebook 00)")

FOTO = fotos[0]
r = detector.predict(FOTO, conf=.25, classes=[CLASSE_GARRAFA], verbose=False)[0]
print(f"garrafas encontradas: {len(r.boxes)}")

plt.figure(); plt.imshow(cv2.cvtColor(r.plot(line_width=4), cv2.COLOR_BGR2RGB))
plt.axis("off"); plt.title(f"{len(r.boxes)} garrafas — sem treinar nada")
plt.tight_layout(); plt.show()

## Etapa 2 — ensinar o que é "aberta" e "lacrada"

Aqui não existe anotação, caixa desenhada nem ferramenta especial: **duas
pastas**. O nome da pasta é o gabarito.

Treino curto de propósito — o objetivo é a **curva**, não o recorde.

In [ ]:
from IPython.display import clear_output
import matplotlib.pyplot as plt

def treinar_mostrando(modelo, dados, epocas, imgsz=224, batch=32,
                      projeto="/content/runs", nome="ao_vivo", titulo="Aprendendo"):
    """Treina e redesenha a curva a cada epoca — o ponto alto da demo."""
    hist = {}                                  # epoca -> (perda, acuracia)

    def a_cada_epoca(trainer):
        m = getattr(trainer, "metrics", None) or {}
        acc = m.get("metrics/accuracy_top1")
        perda = float(trainer.loss.item()) if getattr(trainer, "loss", None) is not None else None
        hist[trainer.epoch + 1] = (perda, acc)   # dict: a epoca repetida sobrescreve

        eps = sorted(hist)
        perdas = [hist[e][0] for e in eps]
        accs = [(hist[e][1] or 0) * 100 for e in eps]

        clear_output(wait=True)
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(20, 8))
        a1.plot(eps, perdas, lw=5, color="#ff5c5c", marker="o", ms=10)
        a1.set_title("ERRO — tem que descer"); a1.set_xlabel("época")
        a2.plot(eps, accs, lw=5, color="#3fe0a8", marker="o", ms=10)
        a2.set_ylim(0, 101)
        a2.set_title("ACERTO — tem que subir"); a2.set_xlabel("época"); a2.set_ylabel("%")
        if accs:
            a2.text(eps[-1], accs[-1], f"  {accs[-1]:.0f}%", fontsize=34,
                    color="#3fe0a8", va="center", fontweight="bold")
        fig.suptitle(f"{titulo} · época {max(eps)} de {epocas}", fontsize=34)
        plt.tight_layout(); plt.show()

    modelo.add_callback("on_fit_epoch_end", a_cada_epoca)
    r = modelo.train(data=dados, epochs=epocas, imgsz=imgsz, batch=batch,
                     project=projeto, name=nome, exist_ok=True, verbose=False, plots=True)
    print("pesos e graficos em:", r.save_dir)
    return r

In [ ]:
EPOCAS = 25          # ← mexa aqui no palco: 5 aprende pouco, 25 já separa bem

especialista = YOLO(f"{DRIVE}/00-pesos/yolo11n-cls.pt")
res = treinar_mostrando(
    especialista,
    dados=f"{DRIVE}/04-garrafas/treino",
    epocas=EPOCAS,
    titulo="Aprendendo a diferença entre aberta e lacrada",
    nome="garrafas",
)

In [ ]:
# ── guarda os pesos no Drive: treinou uma vez, usa para sempre ──
import shutil
origem = f"{res.save_dir}/weights/best.pt"
destino = f"{DRIVE}/04-garrafas/pesos/garrafas_best.pt"
shutil.copy(origem, destino)
print("especialista salvo em", destino)

## Etapa 3 — os dois juntos: o inventário

In [ ]:
# ── detector acha cada garrafa · especialista julga cada uma ──
import cv2, numpy as np, matplotlib.pyplot as plt
from collections import Counter

especialista = YOLO(f"{DRIVE}/04-garrafas/pesos/garrafas_best.pt")
img = cv2.imread(FOTO)
r = detector.predict(FOTO, conf=.25, classes=[CLASSE_GARRAFA], verbose=False)[0]

placar = Counter()
anotada = img.copy()
CORES = {"lacrada": (168, 224, 63), "aberta": (92, 92, 255)}

for caixa in r.boxes.xyxy.cpu().numpy().astype(int):
    x1, y1, x2, y2 = caixa
    recorte = img[max(0, y1):y2, max(0, x1):x2]
    if recorte.size == 0:
        continue
    p = especialista.predict(recorte, verbose=False)[0]
    rotulo = p.names[int(p.probs.top1)]
    certeza = float(p.probs.top1conf)
    placar[rotulo] += 1
    cor = CORES.get(rotulo, (200, 200, 200))
    cv2.rectangle(anotada, (x1, y1), (x2, y2), cor, 4)
    cv2.putText(anotada, f"{rotulo} {certeza:.0%}", (x1, max(28, y1 - 12)),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, cor, 3)

print("INVENTARIO")
for k, v in placar.most_common():
    print(f"  {v:>3}  {k}")

plt.figure(); plt.imshow(cv2.cvtColor(anotada, cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.title(f"Estoque: {sum(placar.values())} garrafas · " +
          " · ".join(f"{v} {k}" for k, v in placar.most_common()))
plt.tight_layout(); plt.show()
cv2.imwrite(f"{DRIVE}/04-garrafas/saida/inventario.jpg", anotada)

In [ ]:
# ── o painel de estoque, em tamanho de palco ──
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(16, 7))
itens = placar.most_common()
ax.bar([i[0] for i in itens], [i[1] for i in itens],
       color=[VERDE if i[0] == "lacrada" else VERMELHO for i in itens], width=.55)
for i, (k, v) in enumerate(itens):
    ax.text(i, v + .08, str(v), ha="center", fontsize=44, fontweight="bold",
            color=VERDE if k == "lacrada" else VERMELHO)
ax.set_title(f"Inventário automático · {sum(placar.values())} garrafas")
ax.set_ylabel("unidades"); ax.set_ylim(0, max(placar.values()) * 1.25)
plt.tight_layout(); plt.show()